# Symptom Extraction from User Input (NLP)
This notebook demonstrates symptom extraction from user input using an LLM.

In [1]:
%pip install groq python-dotenv openai

  Using cached jiter-0.13.0-cp312-cp312-win_amd64.whl.metadata (5.3 kB)
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.1 MB ? eta -:--:--
   --------------------------- ------------ 0.8/1.1 MB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 1.9 MB/s eta 0:00:00
Using cached jiter-0.13.0-cp312-cp312-win_amd64.whl (205 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Configuration

The symptom extraction pipeline was tested using both local and cloud-based LLM models:

- **llama3.2:3b** (local)
- **llama3:8b** (local)
- **qwen2.5:14b** (local)
- **meta-llama/llama-4-scout-17b-16e-instruct** (cloud)
- **openai/gpt-oss-120b** (cloud)

The selection criteria and evaluation results are described in **nlp-test**.

## Option 1 (cloud LLM)

In [3]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("LLM_API_KEY")

# Configure Groq
client = Groq(api_key=groq_api_key)

MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"

## Option 2 (local LLM)

In [4]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

MODEL = "llama3:8"

## System Prompt

The LLM is guided by a structured system prompt designed for **medical symptom extraction and normalization**.

The prompt instructs the model to:

- Extract only **explicitly mentioned** symptoms — no inferences or diagnoses
- Separate symptoms into **present** and **absent** lists
- Normalize terms to **canonical, ontology-compatible** forms (SYMP/DOID)
- Preserve clinically relevant modifiers (e.g., *dry cough*, *high fever*)
- Return output exclusively as **valid JSON**

In [5]:
from pathlib import Path

BASE_DIR = Path.cwd()
prompt_path = BASE_DIR / "tests" / "nlp-extraction-test" / "prompts" / "symptom-extraction-prompt-1"

SYSTEM_PROMPT = prompt_path.read_text(encoding="utf-8")

## Symptom Extraction Function

The `extract_symptoms` function sends patient free-text to the LLM and returns two lists of normalized symptom terms.

**Parameters:**
- `user_input` — raw patient input text

**Returns:**
- `present` — list of symptoms the patient explicitly reports
- `absent` — list of symptoms the patient explicitly denies

**Key configuration:**
- `temperature=0.0` and `seed=42` ensure fully deterministic, reproducible outputs
- `max_tokens=1500` is sufficient given the structured JSON output format
- On any API or parsing error, the function returns two empty lists to prevent pipeline failure

In [6]:
import json
from typing import List, Tuple

def extract_symptoms(user_input: str) -> Tuple[List[str], List[str]]:
    try:
        completion = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_input}
            ],
            temperature=0.0,
            top_p=1,
            max_tokens=1500,
            seed=42,
            stream=False,
        )

        result = json.loads(completion.choices[0].message.content)
        present = result.get("present", [])
        absent  = result.get("absent", [])
        return present, absent

    except Exception as e:
        print(f"Error: {e}")
        return [], []

## Test Data

Loading test data used for evaluation. Each test case is represented as a JSON object with the following structure:

- `id` — unique identifier for the test case (e.g., `CANONICAL_001`)
- `input` — raw patient free-text input
- `expected` — list of canonical symptom terms expected to be extracted
- `synonyms` — dictionary mapping each expected term to its accepted medical synonyms (used for flexible evaluation)

**Example:**
```json
{
    "id": "CANONICAL_001",
    "input": "I have a headache and a fever.",
    "expected": ["headache", "fever"],
    "synonyms": {
        "fever": ["pyrexia"],
        "headache": ["cephalalgia"]
    }
}
```

In [7]:
test_cases_path = BASE_DIR / "tests" / "nlp-extraction-test" / "symptom-extraction-test-data-en.json"

with open(test_cases_path, "r", encoding="utf-8") as f:
    test_cases = json.load(f)

## Evaluation Function

The `evaluate_case` function compares extracted symptoms against expected values and computes standard information retrieval metrics.

### Normalization
All terms are lowercased and stripped of whitespace before comparison to avoid case-sensitivity mismatches.

### Synonym Resolution
Before scoring, extracted terms are resolved against a synonym dictionary. If an extracted term is a known synonym of an expected canonical term (e.g., `pyrexia` → `fever`), it is treated as a correct match.

### Metrics
Standard precision, recall and F1 are computed over the resolved sets:

- **Precision** — fraction of extracted symptoms that are correct
- **Recall** — fraction of expected symptoms that were successfully extracted
- **F1** — harmonic mean of precision and recall

### Negation Error Detection
If any symptom from the `absent` list (symptoms the patient denied) maps to a term in the `expected` list, a `negation_error` flag is set to `True` — indicating the model incorrectly extracted a negated symptom as present.

### Output
```json
{
    "precision": 0.857,
    "recall": 1.000,
    "f1": 0.923,
    "false_positive": ["neck stiffness"],
    "false_negative": [],
    "negation_error": false
}
```

In [8]:
def normalize(text: str) -> str:
    if not text:
        return ""
    return text.lower().strip()


def evaluate_case(extracted, expected, synonyms=None, not_expected=None):
    extracted_norm = set(normalize(x) for x in extracted) if extracted else set()
    expected_norm  = set(normalize(x) for x in expected)  if expected  else set()

    synonym_to_canonical = {}
    if synonyms:
        for canonical, syns in synonyms.items():
            canonical_norm = normalize(canonical)
            for syn in syns:
                synonym_to_canonical[normalize(syn)] = canonical_norm

    resolved = set()
    for term in extracted_norm:
        if term in expected_norm:
            resolved.add(term)
        elif term in synonym_to_canonical:
            canonical = synonym_to_canonical[term]
            if canonical in expected_norm:
                resolved.add(canonical)
        else:
            resolved.add(term)

    if len(resolved) == 0 and len(expected_norm) == 0:
        return {
            "precision": 1.0,
            "recall":    1.0,
            "f1":        1.0,
            "false_positive": [],
            "false_negative": [],
            "negation_error": False
        }

    true_positive  = resolved & expected_norm
    false_positive = resolved - expected_norm
    false_negative = expected_norm - resolved

    precision = len(true_positive) / len(resolved)      if resolved      else 0.0
    recall    = len(true_positive) / len(expected_norm)  if expected_norm else 0.0
    f1        = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    negation_error = False
    if not_expected:
        not_expected_norm = {normalize(item) for item in not_expected}

        not_expected_resolved = set()
        for term in not_expected_norm:
            if term in synonym_to_canonical:
                not_expected_resolved.add(synonym_to_canonical[term])
            else:
                not_expected_resolved.add(term)
        if not_expected_resolved & expected_norm:
            negation_error = True

    return {
        "precision":     round(precision, 3),
        "recall":        round(recall, 3),
        "f1":            round(f1, 3),
        "false_positive": sorted(list(false_positive)),
        "false_negative": sorted(list(false_negative)),
        "negation_error": negation_error
    }


## Evaluation Pipeline

The `test_symptom_extraction_to_file` function runs the full evaluation loop over all test cases and saves the results to a `.txt` report file.

### Output File
Results are saved to `tests/nlp-extraction-test/results/` with a timestamp header and the model name, providing a reproducible record of each evaluation run.

### Retry Logic
Each test case supports up to 5 retries with exponential backoff (`2^n` seconds) to handle API rate limiting (HTTP 429). Non-rate-limit errors result in an immediate empty response to avoid blocking the pipeline.

### Per-Case Output
For each test case the following is logged:
- Raw input text
- Extracted present and absent symptom lists
- Expected symptom list
- Precision, Recall and F1 scores
- False positives and false negatives
- Negation error flag

### Rate Limiting
A fixed 5-second delay is applied between API calls (`time.sleep(5)`) to stay within API usage limits.

### Summary
After all cases are processed, aggregate averages for Precision, Recall and F1 are computed and appended to the report, along with a list of any failed case IDs.

In [9]:
import os
import time
import statistics
from datetime import datetime

def test_symptom_extraction_to_file(max_retries=5, sleep_base=2):
    results_dir = "tests/nlp-extraction-test/results"
    file_name = "symptom-extraction-result-5.txt"

    if not os.path.exists(results_dir):
        os.makedirs(results_dir)

    full_path = os.path.join(results_dir, file_name)

    total_precision = []
    total_recall    = []
    total_f1        = []
    failed_cases    = []

    with open(full_path, "w", encoding="utf-8") as f:
        f.write(f"=== SYMPTOM EXTRACTION REPORT - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} ===\n")
        f.write(f"Model: {MODEL}\n")
        f.write("====================================================\n\n")

        for i, case in enumerate(test_cases):
            present, absent = None, None
            retry_count = 0

            while retry_count < max_retries:
                try:
                    present, absent = extract_symptoms(case["input"])
                    break
                except Exception as e:
                    err_str = str(e)
                    msg = f"Error on {case['id']}: {err_str}"
                    print(msg)
                    f.write(msg + "\n")

                    if "429" in err_str or "rate limit" in err_str.lower():
                        retry_count += 1
                        wait_time = sleep_base ** retry_count
                        time.sleep(wait_time)
                    else:
                        present, absent = [], []
                        break

            if present is None:
                present, absent = [], []
                failed_cases.append(case["id"])

            result = evaluate_case(
                extracted    = present,
                expected     = case["expected"],
                synonyms     = case.get("synonyms"),
                not_expected = absent
            )

            total_precision.append(result["precision"])
            total_recall.append(result["recall"])
            total_f1.append(result["f1"])

            case_output = [
                f"--- {case['id']} ---",
                f"Input: {case['input']}",
                f"Present (extracted): {sorted(present)}",
                f"Absent (negated):    {sorted(absent)}",
                f"Expected:           {sorted(case['expected'])}",
                f"Precision: {result['precision']:.3f} | Recall: {result['recall']:.3f} | F1: {result['f1']:.3f}",
                f"False Positive: {result['false_positive']}",
                f"False Negative: {result['false_negative']}",
                f"Negation Error: {result['negation_error']}",
                "\n"
            ]

            output_str = "\n".join(case_output)
            print(output_str)
            f.write(output_str)

            #time.sleep(5)

        avg_precision = statistics.mean(total_precision) if total_precision else 0
        avg_recall    = statistics.mean(total_recall)    if total_recall    else 0
        avg_f1        = statistics.mean(total_f1)        if total_f1        else 0

        summary = [
            "====================",
            f"Average Precision: {avg_precision:.3f}",
            f"Average Recall:    {avg_recall:.3f}",
            f"Average F1-score:  {avg_f1:.3f}",
            f"Failed IDs: {failed_cases}" if failed_cases else "All cases processed."
        ]

        summary_str = "\n".join(summary)
        print("\n" + summary_str)
        f.write("\n" + summary_str)

    print(f"\nResults saved at: {full_path}")

In [7]:
test_symptom_extraction_to_file()

--- CANONICAL_001 ---
Input: I have a headache and a fever.
Present (extracted): ['fever', 'headache']
Absent (negated):    []
Expected:           ['fever', 'headache']
Precision: 1.000 | Recall: 1.000 | F1: 1.000
False Positive: []
False Negative: []
Negation Error: False


--- CANONICAL_002 ---
Input: I have a headache, high fever, and I'm coughing badly.
Present (extracted): ['cough', 'headache', 'high fever']
Absent (negated):    []
Expected:           ['cough', 'headache', 'high fever']
Precision: 1.000 | Recall: 1.000 | F1: 1.000
False Positive: []
False Negative: []
Negation Error: False


--- CANONICAL_003 ---
Input: I've been struggling with nausea, vomiting, and diarrhea for two days.
Present (extracted): ['diarrhea', 'nausea', 'vomiting']
Absent (negated):    []
Expected:           ['diarrhea', 'nausea', 'vomiting']
Precision: 1.000 | Recall: 1.000 | F1: 1.000
False Positive: []
False Negative: []
Negation Error: False


--- CANONICAL_004 ---
Input: Joint pain, fever, and a 